# Music Annotation Project V8.4 - Notebook 42
## Failure-mode, semantic-breadth and visual audit

**Runtime: CPU.** This is a read-only diagnostic of Notebook 41. It does not retrain, recalibrate, move thresholds, alter selections or authorise promotion.

It compares Page003 false-safe groups 25/26 with correctly selected multiscale groups, tests whether contrasts generalise to missed-benefit groups outside `Anglo_Music_Press_HotH`, and completes `V4_9_semantic_breadth` and `V4_10_visual_audit` through a blinded Gradio review.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy scikit-learn gradio pillow

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,math,shutil,sys,platform,zipfile
import cv2,numpy as np,pandas as pd,gradio as gr
from scipy.stats import mannwhitneyu
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut,cross_val_predict
from sklearn.metrics import average_precision_score,roc_auc_score
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject'); V84_ROOT=ROOT/'observed_print_classifier_v8_4'
SOURCE_V84_RUN_OVERRIDE=None; SEED=42001; REVIEW_MARGIN=180
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook42_v84_audit'; OUT=ROOT/'observed_print_classifier_v8_4_notebook42'/RUN
TABLES=OUT/'tables';CROPS=OUT/'crops';REVIEW=OUT/'review';UPLOAD=OUT/'upload'
for p in [OUT,TABLES,CROPS,REVIEW,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def latest(root,req):
 hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in req)] if Path(root).exists() else []
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
REQ=['14_experiment_manifest.json','15_v8_4_verdict.json','candidate_lock.json','tables/04_oof_safety_and_value_predictions.parquet','tables/11_visual_review_queue.csv']
SOURCE=Path(SOURCE_V84_RUN_OVERRIDE) if SOURCE_V84_RUN_OVERRIDE else latest(V84_ROOT,REQ)
if SOURCE is None:raise RuntimeError('No complete Notebook 41 run found.')
manifest=json.loads((SOURCE/'14_experiment_manifest.json').read_text()); verdict=json.loads((SOURCE/'15_v8_4_verdict.json').read_text())
record={'source':str(SOURCE),'manifest_sha256':sha(SOURCE/'14_experiment_manifest.json'),'verdict_sha256':sha(SOURCE/'15_v8_4_verdict.json'),'candidate_lock_sha256':sha(SOURCE/'candidate_lock.json')}
(OUT/'source_record.json').write_text(json.dumps(record,indent=2));print(json.dumps(record,indent=2))

## Build the audit cohorts and recover observable spatial context

The diagnostic descriptors below are post-hoc summaries only. They are not new gates or deterministic continuation rules.

In [ ]:
oof=pd.read_parquet(SOURCE/'tables/04_oof_safety_and_value_predictions.parquet');q=oof[oof.arm=='multiscale_33_97'].copy().reset_index(drop=True)
q['audit_cohort']=np.select([q.selected&(q.safety_label==0),q.selected&(q.safety_label==1)&(q.value_score>0),q.selected&(q.value_score<=0),(~q.selected)&(q.safety_label==1)&(q.value_score>0)&(q.source_group!='Anglo_Music_Press_HotH'),(~q.selected)&(q.safety_label==1)&(q.value_score>0)&(q.source_group=='Anglo_Music_Press_HotH')],['false_safe','selected_benefit','negative_value','missed_benefit_outside','missed_benefit_inside'],default='other')
focus=q[q.audit_cohort!='other'].copy()
if not {'Page003:g00025','Page003:g00026'}.issubset(set(focus.group_id)):raise RuntimeError('Expected Page003 false-safe groups not found.')
source_v82=Path(manifest['source_record']['source']);v75=Path(manifest['source_record']['v75'])
status=pd.read_csv(v75/'tables/cache_status.csv');cache={str(r.page):Path(r.cache) for _,r in status.iterrows()};clean_map={p.stem:p for p in (ROOT/'source'/'clean').iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}
full82=pd.read_parquet(source_v82/'tables/all_eligible_oof_predictions.parquet');ctx={};group_masks={}
for page in sorted(focus.page.unique()):
 with np.load(cache[page]) as z:a={k:z[k].copy() for k in ['mask','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']}
 obs=cv2.imread(str(source_v82/'whole_pages'/page/'observed.png'),0);base=cv2.imread(str(source_v82/'whole_pages'/page/'v7_8.png'),0);clean=cv2.imread(str(clean_map[page]),0);mask=a['mask']>0
 _,labels,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
 ctx[page]={'obs':obs,'base':base,'clean':clean,'mask':mask,'labels':labels,'stats':stats,'edge':cv2.Canny(obs,70,170)>0,**{k.replace('g80_',''):a[k]>0 for k in a if k!='mask'}}
 pg=full82[full82.page==page];pm=np.zeros(obs.shape,np.float32);elig=np.zeros(obs.shape,bool);ys=pg.y.to_numpy(int);xs=pg.x.to_numpy(int);pm[ys,xs]=pg.p_print.to_numpy();elig[ys,xs]=True
 _,labs,_,_=cv2.connectedComponentsWithStats((elig&(pm>=.50)&(pm<.75)).astype(np.uint8),8);group_masks[page]=labs
def frow(r):
 d=ctx[r.page];gm=group_masks[r.page]==int(r.candidate_group_id)
 if not gm.any():raise RuntimeError(f'{r.group_id}: group identity not rebuilt')
 y,x=np.where(gm);x0,x1=x.min(),x.max()+1;y0,y1=y.min(),y.max()+1;ring=(cv2.dilate(gm.astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(17,17)))>0)&~gm;wide=(cv2.dilate(gm.astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(65,65)))>0)&~gm;dark=d['obs']<190
 contacts=[]
 for dy,dx in [(0,-1),(0,1),(-1,0),(1,0)]:
  z=np.roll(gm,(dy,dx),(0,1))&~gm;contacts.append(float(dark[z].mean()) if z.any() else 0.)
 return {'group_id':r.group_id,'bbox_w':x1-x0,'bbox_h':y1-y0,'aspect_ratio':max(x1-x0,y1-y0)/max(min(x1-x0,y1-y0),1),'fill_ratio':gm.sum()/max((x1-x0)*(y1-y0),1),'observed_mean':float(d['obs'][gm].mean()),'observed_std':float(d['obs'][gm].std()),'ring_dark_rate':float(dark[ring].mean()),'wide_dark_rate':float(dark[wide].mean()),'ring_edge_rate':float(d['edge'][ring].mean()),'wide_edge_rate':float(d['edge'][wide].mean()),'horizontal_ring':float(d['horizontal'][ring].mean()),'vertical_ring':float(d['vertical'][ring].mean()),'compact_ring':float(d['compact'][ring].mean()),'preserve_ring':float(d['preserve'][ring].mean()),'remove_ring':float(d['remove'][ring].mean()),'ambiguous_ring':float(d['ambiguous'][ring].mean()),'two_sided_horizontal_contact':min(contacts[:2]),'two_sided_vertical_contact':min(contacts[2:])}
feat=pd.DataFrame([frow(r) for _,r in focus.iterrows()]);audit=focus.merge(feat,on='group_id',validate='one_to_one');audit.to_parquet(TABLES/'observable_pattern_features.parquet',index=False);focus.to_csv(TABLES/'audit_cohorts.csv',index=False)
print(focus.audit_cohort.value_counts())

In [ ]:
FEATURES=[c for c in feat if c!='group_id']+[c for c in ['pixels','p_mean','p_min','p_max','p_std','mask_depth_mean','edge_rate','horizontal_rate','vertical_rate','compact_rate','ambiguous_rate','p_safety','p_value','pred_value'] if c in audit]
rows=[]
def contrast(a,b,name):
 A=audit[audit.audit_cohort==a];B=audit[audit.audit_cohort==b]
 for f in FEATURES:
  x=A[f].astype(float).dropna();y=B[f].astype(float).dropna()
  if len(x)<2 or len(y)<2:continue
  _,p=mannwhitneyu(x,y,alternative='two-sided');sd=np.sqrt(((len(x)-1)*x.var(ddof=1)+(len(y)-1)*y.var(ddof=1))/max(len(x)+len(y)-2,1));d=(x.mean()-y.mean())/sd if sd>0 else np.nan
  rows.append({'comparison':name,'feature':f,'n_a':len(x),'n_b':len(y),'mean_a':x.mean(),'mean_b':y.mean(),'standardised_difference':d,'mannwhitney_p':p})
contrast('false_safe','selected_benefit','false_safe_vs_selected_benefit');contrast('selected_benefit','missed_benefit_outside','selected_benefit_vs_outside_missed')
cmp=pd.DataFrame(rows);cmp['abs_difference']=cmp.standardised_difference.abs();cmp.sort_values(['comparison','abs_difference'],ascending=[True,False]).to_csv(TABLES/'feature_contrasts.csv',index=False)
base=['pixels','p_mean','p_min','p_max','p_std','mask_depth_mean','edge_rate','horizontal_rate','vertical_rate','compact_rate','ambiguous_rate','p_safety','p_value','pred_value'];m=q[(q.safety_label==1)&q.value_positive.isin([0,1])].copy();X=m[base].astype(float);y=m.value_positive.astype(int);g=m.source_group.astype(str)
if g.nunique()>=3 and y.nunique()==2:
 pred=cross_val_predict(Pipeline([('scale',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=5000,random_state=SEED))]),X,y,groups=g,cv=LeaveOneGroupOut(),method='predict_proba')[:,1]
 outside=g!='Anglo_Music_Press_HotH';gen={'rows':len(m),'source_groups':int(g.nunique()),'average_precision':float(average_precision_score(y,pred)),'roc_auc':float(roc_auc_score(y,pred)),'outside_anglo_average_precision':float(average_precision_score(y[outside],pred[outside])) if y[outside].nunique()==2 else None}
else:gen={'status':'insufficient variation'}
(OUT/'generalisation_diagnostic.json').write_text(json.dumps(gen,indent=2));print(cmp.sort_values('abs_difference',ascending=False).head(12).to_string(index=False));print(json.dumps(gen,indent=2))

## Create the semantic-breadth queue and blinded review panels

Review all false-safe and selected groups, plus the strongest missed-benefit cases sampled across source groups. The blinded panel shows observed and V7.8 first; clean truth is revealed only as a diagnostic.

In [ ]:
def morphology(r):
 if r.aspect_ratio>=3 and r.two_sided_horizontal_contact>=.15:return 'horizontal_linear'
 if r.aspect_ratio>=3 and r.two_sided_vertical_contact>=.15:return 'vertical_linear'
 if r.compact_ring>=.10 or (r.bbox_w<=8 and r.bbox_h<=8):return 'compact_fragment'
 if r.ring_edge_rate>=.15 and r.wide_dark_rate>=.12:return 'dense_intersection'
 if r.ambiguous_ring>=.20:return 'ambiguous_overlap'
 return 'other_fragment'
audit['provisional_morphology']=audit.apply(morphology,axis=1)
queue=pd.concat([audit[audit.audit_cohort.isin(['false_safe','selected_benefit','negative_value'])],audit[audit.audit_cohort=='missed_benefit_outside'].sort_values('value_score',ascending=False).groupby('source_group').head(4)]).drop_duplicates('group_id')
for c,v in {'review_status':'unreviewed','semantic_class':'','visual_outcome':'','systematic_failure':'','review_notes':''}.items():queue[c]=v
queue.to_csv(REVIEW/'review_queue.csv',index=False);audit.to_csv(TABLES/'semantic_breadth_population.csv',index=False)
def panel(r,truth=False):
 d=ctx[r.page];cy=int(round(r.centre_y));cx=int(round(r.centre_x));x0=max(0,cx-REVIEW_MARGIN);x1=min(d['obs'].shape[1],cx+REVIEW_MARGIN+1);y0=max(0,cy-REVIEW_MARGIN);y1=min(d['obs'].shape[0],cy+REVIEW_MARGIN+1)
 obs=d['obs'][y0:y1,x0:x1];base=d['base'][y0:y1,x0:x1];clean=d['clean'][y0:y1,x0:x1];rgb=cv2.cvtColor(obs,cv2.COLOR_GRAY2BGR);cv2.circle(rgb,(cx-x0,cy-y0),8,(0,0,255),2);ims=[rgb,cv2.cvtColor(base,cv2.COLOR_GRAY2BGR)];titles=['OBSERVED','V7.8']
 if truth:ims.append(cv2.cvtColor(clean,cv2.COLOR_GRAY2BGR));titles.append('CLEAN TRUTH')
 h=max(i.shape[0] for i in ims);ims=[cv2.copyMakeBorder(i,0,h-i.shape[0],0,0,cv2.BORDER_CONSTANT,value=(255,255,255)) for i in ims];out=np.hstack(ims);canvas=np.full((h+52,out.shape[1],3),255,np.uint8);canvas[52:]=out;cv2.putText(canvas,str(r.group_id),(8,21),cv2.FONT_HERSHEY_SIMPLEX,.55,(30,30,30),1,cv2.LINE_AA);w=out.shape[1]//len(ims)
 for i,t in enumerate(titles):cv2.putText(canvas,t,(i*w+8,44),cv2.FONT_HERSHEY_SIMPLEX,.46,(30,30,30),1,cv2.LINE_AA)
 return canvas
for _,r in queue.iterrows():
 n=r.group_id.replace(':','_');cv2.imwrite(str(CROPS/f'{n}_blinded.png'),panel(r,False));cv2.imwrite(str(CROPS/f'{n}_diagnostic.png'),panel(r,True))
print('Queue rows:',len(queue),'source groups:',queue.source_group.nunique(),'morphologies:',queue.provisional_morphology.nunique())

In [ ]:
SEMANTIC=['staff_line','stem_or_barline','beam_or_flag','notehead_or_compact_glyph','slur_tie_or_curve','text_or_annotation_stroke','mixed_print_annotation_intersection','background_or_tonal','registration_ambiguous','other'];OUTCOMES=['faithful_benefit','false_safe_annotation','negative_or_no_benefit','missed_benefit','indeterminate'];qp=REVIEW/'review_queue.csv'
def load():return pd.read_csv(qp).fillna('')
def view(gid,truth):
 r=load().query('group_id==@gid').iloc[0];p=CROPS/(gid.replace(':','_')+('_diagnostic.png' if truth else '_blinded.png'));info=f'{gid} | {r.page} | {r.source_group} | morphology {r.provisional_morphology} | p_safety {float(r.p_safety):.4f} | p_value {float(r.p_value):.4f}'
 return str(p),info,r.semantic_class or None,r.visual_outcome or None,r.systematic_failure or 'no',r.review_notes
def save_review(gid,semantic,outcome,systematic,notes):
 d=load();m=d.group_id==gid;d.loc[m,['semantic_class','visual_outcome','systematic_failure','review_notes','review_status']]=[semantic,outcome,systematic,notes,'reviewed'];d.to_csv(qp,index=False);return f'Saved. Reviewed {(d.review_status=="reviewed").sum()}/{len(d)}.'
ids=load().group_id.tolist()
with gr.Blocks(title='Notebook 42 review') as app:
 gr.Markdown('# Notebook 42 - semantic breadth and visual audit')
 gid=gr.Dropdown(ids,value=ids[0],label='Group');truth=gr.Checkbox(False,label='Reveal clean truth after initial review');im=gr.Image(height=650);info=gr.Textbox(interactive=False);sem=gr.Dropdown(SEMANTIC,label='Semantic class');out=gr.Dropdown(OUTCOMES,label='Visual outcome');systematic=gr.Radio(['no','yes','indeterminate'],value='no',label='Unmodelled systematic failure?');notes=gr.Textbox(label='Notes');save=gr.Button('Save',variant='primary');status=gr.Textbox(interactive=False)
 gid.change(view,[gid,truth],[im,info,sem,out,systematic,notes]);truth.change(view,[gid,truth],[im,info,sem,out,systematic,notes]);save.click(save_review,[gid,sem,out,systematic,notes],status);app.load(view,[gid,truth],[im,info,sem,out,systematic,notes])
app.launch(share=True,debug=False,show_error=True)

## Finalise V4_9 and V4_10

Run this cell after closing the reviewer. The gates fail closed if the queue is incomplete, semantic breadth is too narrow, source diversity is inadequate, either Page003 false-safe case is unreviewed, or any reviewed case records a systematic failure.

In [ ]:
r=pd.read_csv(REVIEW/'review_queue.csv').fillna('');done=r.review_status.eq('reviewed');sc=r[done].groupby('semantic_class').size();sg=r[done].groupby('source_group').size();co=r[done].groupby('audit_cohort').size();required=set(r.loc[r.audit_cohort=='false_safe','group_id']);reviewed_false=set(r.loc[done&(r.audit_cohort=='false_safe'),'group_id'])
V49=bool(done.sum()>=20 and sc.size>=4 and sg.size>=3 and required.issubset(reviewed_false));V410=bool(done.all() and {'false_safe','selected_benefit','missed_benefit_outside'}.issubset(set(co.index)) and not r.loc[done,'systematic_failure'].eq('yes').any())
result={'V4_9_semantic_breadth':V49,'V4_10_visual_audit':V410,'reviewed':int(done.sum()),'queue_rows':len(r),'semantic_classes':int(sc.size),'reviewed_source_groups':int(sg.size),'systematic_failure_yes':int(r.loc[done,'systematic_failure'].eq('yes').sum())}
sc.rename('reviewed').to_csv(TABLES/'semantic_class_counts.csv');sg.rename('reviewed').to_csv(TABLES/'reviewed_source_counts.csv');co.rename('reviewed').to_csv(TABLES/'reviewed_cohort_counts.csv');r[r.group_id.isin(['Page003:g00025','Page003:g00026'])].to_csv(TABLES/'page003_false_safe_25_26_review.csv',index=False)
(OUT/'notebook42_gate_completion.json').write_text(json.dumps(result,indent=2));(OUT/'v8_4_verdict_with_notebook42_review.json').write_text(json.dumps({'source_notebook41_verdict':verdict,'notebook42_completion':result,'development_only':True},indent=2))
cmp=pd.read_csv(TABLES/'feature_contrasts.csv');a=cmp[cmp.comparison=='false_safe_vs_selected_benefit'].nlargest(8,'abs_difference');b=cmp[cmp.comparison=='selected_benefit_vs_outside_missed'].nlargest(8,'abs_difference')
summary='NOTEBOOK 42 - V8.4 AUDIT\n\n'+json.dumps(result,indent=2)+'\n\nFALSE-SAFE VS SELECTED BENEFIT\n'+a[['feature','mean_a','mean_b','standardised_difference','mannwhitney_p']].to_string(index=False)+'\n\nSELECTED BENEFIT VS OUTSIDE MISSED BENEFIT\n'+b[['feature','mean_a','mean_b','standardised_difference','mannwhitney_p']].to_string(index=False)+'\n\nPost-hoc diagnostic evidence only; no threshold or routing rule is authorised.'
(OUT/'executive_summary.txt').write_text(summary);identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook42-v8.4-audit','source_record':record,'development_only':True,'retrained':False,'recalibrated':False,'gate_completion':result};(OUT/'experiment_manifest.json').write_text(json.dumps(identity,indent=2))
files=[OUT/'source_record.json',OUT/'generalisation_diagnostic.json',OUT/'notebook42_gate_completion.json',OUT/'v8_4_verdict_with_notebook42_review.json',OUT/'executive_summary.txt',OUT/'experiment_manifest.json',TABLES/'audit_cohorts.csv',TABLES/'observable_pattern_features.parquet',TABLES/'feature_contrasts.csv',TABLES/'semantic_breadth_population.csv',TABLES/'semantic_class_counts.csv',TABLES/'reviewed_source_counts.csv',TABLES/'reviewed_cohort_counts.csv',TABLES/'page003_false_safe_25_26_review.csv',REVIEW/'review_queue.csv']
inv=[]
for p in files:
 if not p.exists():raise RuntimeError(f'Missing upload artefact: {p}')
 dst=UPLOAD/p.relative_to(OUT);dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst);inv.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha(p)})
for _,x in r[done].iterrows():
 for suf in ['_blinded.png','_diagnostic.png']:
  p=CROPS/(x.group_id.replace(':','_')+suf);dst=UPLOAD/'crops'/p.name;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst);inv.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'README.txt').write_text('Start with executive_summary.txt, notebook42_gate_completion.json, tables/feature_contrasts.csv and review/review_queue.csv.')
zip_path=UPLOAD/'notebook42_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file() and p!=zip_path:z.write(p,p.relative_to(UPLOAD))
print(summary);print('Upload bundle:',zip_path)